# Explainable Root Cause Analysis for (GNN) Traffic Prediction Failures

Pipeline: Load → Preprocess → Baseline → Explainability (SHAP, LIME, counterfactuals) → RCA (DoWhy, sensitivity, causal graph) → Evaluate & visualize.

In [ ]:
import os, sys
ROOT = os.getcwd()  # Run notebook from project root (Graphs folder)
sys.path.insert(0, ROOT)
os.chdir(ROOT)
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import config
from data.load_and_preprocess import load_abilene_like_data, preprocess_dataset, get_train_test_split
from models.baseline import train_baseline_model, get_baseline_predictions
from explainability.feature_importance import get_model_importance, get_permutation_importance
from explainability.shap_explanations import compute_shap_values, shap_summary_plot
from explainability.lime_explanations import get_lime_explanations
from explainability.counterfactuals import generate_counterfactuals
from rca.root_cause import identify_key_drivers, run_causal_analysis, sensitivity_analysis
from rca.causal_graph import build_and_plot_causal_graph
from evaluation.metrics import compute_standard_metrics, compute_interpretability_metrics
from visualization.plots import plot_confusion_matrix, plot_roc_curve, plot_feature_importance
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Load and preprocess

In [ ]:
df, graph, feature_cols, target_name = load_abilene_like_data(data_path=None, n_timesteps=config.N_TIMESTAMPS)
print(f'Samples: {len(df)}, Features: {len(feature_cols)}, Target: {target_name}')
X, y, feature_names, scaler = preprocess_dataset(df, feature_cols, target_name, impute=True, normalize=True)
X_train, X_test, y_train, y_test = get_train_test_split(X, y, stratify=y)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

## 2. Train baseline & evaluate

In [ ]:
model, task = train_baseline_model(X_train, y_train, task='classification', model_type=config.BASELINE_MODEL)
y_pred, y_proba = get_baseline_predictions(model, X_test, task)
metrics = compute_standard_metrics(y_test, y_pred, y_proba, task=task)
print('Test metrics:', metrics)
plot_confusion_matrix(y_test, y_pred, save_path=os.path.join(config.OUTPUT_DIR, 'confusion_matrix.png'))
plt.show()
if y_proba is not None:
    proba_pos = y_proba[:, 1] if getattr(y_proba, 'ndim', 0) > 1 else np.array(y_proba).ravel()
    plot_roc_curve(y_test, proba_pos, save_path=os.path.join(config.OUTPUT_DIR, 'roc_curve.png'))
    plt.show()

## 3. Explainability: feature importance, SHAP, LIME, counterfactuals

In [ ]:
model_imp = get_model_importance(model, feature_names)
perm_imp_df = get_permutation_importance(model, X_test, y_test, n_repeats=5, scoring='accuracy')
plot_feature_importance(model_imp, top_k=15, save_path=os.path.join(config.OUTPUT_DIR, 'feature_importance.png'))
plt.show()
print('Top 5 model importance:', list(model_imp.items())[:5])

In [ ]:
n_shap = min(config.SHAP_SAMPLES, len(X_test))
X_explain = X_test.sample(n_shap, random_state=config.RANDOM_STATE)
shap_vals, _ = compute_shap_values(model, X_train, X_explain, task='classification', max_background=50)
if shap_vals is not None:
    shap_summary_plot(shap_vals, X_explain, feature_names=feature_names, save_path=os.path.join(config.OUTPUT_DIR, 'shap_summary.png'))
    plt.show()

In [ ]:
lime_exp, lime_weights = get_lime_explanations(model, X_test, 0, feature_names=feature_names, task='classification')
if lime_weights:
    print('LIME top features (instance 0):', lime_weights[:5])
fail_idx = np.where(y_test == 1)[0]
if len(fail_idx) > 0:
    cf_df, cf_results = generate_counterfactuals(model, X_test.iloc[fail_idx[0]], feature_names, target_class=0, n_cf=config.N_COUNTERFACTUALS)
    print('Counterfactual example (first 3 features):')
    print(cf_df.iloc[0][:3])

## 4. Root cause analysis: key drivers, DoWhy, sensitivity, causal graph

In [ ]:
key_drivers = identify_key_drivers(model_imp, top_k=10)
print('Key drivers:', [k for k, _ in key_drivers])
sensitivity = sensitivity_analysis(model, X_test, y_test, feature_names, n_perturbations=15, noise_scale=0.1)
plot_feature_importance(dict(sorted(sensitivity.items(), key=lambda x: -x[1])[:15]), top_k=15,
    save_path=os.path.join(config.OUTPUT_DIR, 'sensitivity_analysis.png'), title='Sensitivity analysis')
plt.show()

In [ ]:
df_causal = pd.concat([X_test.reset_index(drop=True), pd.Series(y_test, name=target_name)], axis=1)
treatment = [feature_names[0]] if feature_names else []
common_causes = feature_names[1:4] if len(feature_names) > 3 else (feature_names[1:] if len(feature_names) > 1 else [])
if treatment and common_causes:
    est, _ = run_causal_analysis(df_causal, treatment, target_name, common_causes=common_causes)
    if est is not None:
        print('Causal estimate (DoWhy):', est)

In [ ]:
G_causal, fig = build_and_plot_causal_graph(model_imp, target_name=target_name, top_n=8,
    save_path=os.path.join(config.OUTPUT_DIR, 'causal_graph.png'),
    title='Causal pathways to prediction failure')
plt.show()

## 5. Interpretability metrics

In [ ]:
interp_metrics = compute_interpretability_metrics(model_imp, perm_imp_df, shap_vals, X_explain, feature_names)
print('Interpretability metrics:', interp_metrics)
print('\nOutputs saved to:', config.OUTPUT_DIR)